In [49]:
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict,Annotated
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver
import os


In [50]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

In [51]:
llm = ChatGoogleGenerativeAI(
    model= "gemini-2.5-flash",
    google_api_key=api_key,
)

In [52]:
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]
    

In [53]:
def chat_node(state:ChatState):
    messages=state['messages']

    response=llm.invoke(messages)

    return {'messages':[response]}

In [54]:
checkpointer=MemorySaver()
graph=StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

workflow=graph.compile(checkpointer=checkpointer)


In [56]:
thread_id=1

config={'configurable':{'thread_id':thread_id}}


intial_state={
    'messages':[HumanMessage(content='What is the above question')]
}

workflow.invoke(intial_state,config=config)

{'messages': [HumanMessage(content='What is the capital of pakistan', additional_kwargs={}, response_metadata={}, id='fd7b6d95-374f-41f4-923c-23f79baed069'),
  AIMessage(content='The capital of Pakistan is **Islamabad**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ef388-e9ad-7240-9cbf-eaf422211bd5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 24, 'total_tokens': 31, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 15}}),
  HumanMessage(content='What is the above question', additional_kwargs={}, response_metadata={}, id='576037ec-e68a-4c1c-b96a-c7bd09ab1668'),
  AIMessage(content='The above question was: "What is the capital of pakistan"', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'goo